# detailed working inside the folder
**./4_attention_killer_SSM_mamba**

## The Core Conceptual Problem

Mamba-1 parameterizes selective State Space Models (SSMs) via a 1st-order continuous-time differential equation discretized using Exponential-Euler:

While Mamba-1 achieved $O(T)$ linear-time inference memory, its sequential recurrence was sequential on GPU SRAM ($O(T)$ scan sequential dependency).

## Mamba-2 Bottleneck Duality

1. Hardware ceiling (decoding regime):
   - Memory-bound arithmetic intensity during autoregressive generation ($T=1$).
   - Scalar update $h_t = \alpha_t h_{t-1} + B_t x_t$ yields high bandwidth overhead relative to compute FLOPs (low HBU/MFU on Tensor Cores).

2. Expressivity ceiling (state tracking & recall):
   - Real-valued diagonal decay causes monotone state attenuation.
   - Zero rotational phase degrees of freedom -> inability to solve parity / modular arithmetic / associative memory (induction head emulation).


## Mathematical Derivations & State Dynamics

### A. The Mamba-2 (SSD) Recurrence Formulation

In Mamba-2, the state update for head $h$ at token step $t$ is expressed as:

$$
h_t = \alpha_t h_{t-1} + B_t x_t \quad \in \mathbb{R}^N
$$

$$
y_t = C_t^\top h_t \quad \in \mathbb{R}
$$

where:

- $\alpha_t = \exp(\Delta_t A_t) \in (0, 1]$ is a real scalar decay factor.
- $B_t, C_t \in \mathbb{R}^N$ are data-dependent projection vectors derived from input $x_t$.
- $h_t \in \mathbb{R}^N$ is the latent memory state vector.

## What Problem Is Mamba Trying to Solve?

### The Transformer Bottleneck: $O(T^2)$ KV-Cache Growth

In standard Attention, every single output token must calculate dot-product similarity against all previous tokens.

- Training: Sequence length doubling quadruples computation ($O(T^2)$ matrix size).
- Inference: Every past token's Key and Value vectors must be stored in memory (KV-Cache). Generating token #100,000 requires holding 100,000 vectors in memory for every layer.

## The Old Alternative: Recurrent Neural Networks (RNNs)

RNNs compress all past history into a fixed-size hidden state vector $h_t$.

- Inference: Constant time $O(1)$ memory and time per step. You only keep $h_{t-1}$, update it to $h_t$, and discard the old state.
- The failure: They could not scale or retain long-range information well because they treated all incoming tokens with the exact same transition rules, causing early information to get washed out or blurred over long sequences.

## The Core Idea of Mamba

Mamba combines the $O(1)$ memory footprint of an RNN with the long-range capabilities of Transformers. It achieves this through two main innovations.

### Innovation A: "Selective" Memory (Content-Aware State Updates)

Earlier State Space Models (like S4) used fixed static matrices to update hidden states regardless of what word was being read.

Mamba makes the update parameters a function of the current input token $x_t$.

## Static vs. Selective State Space Models

### Static State Space Model (S4)

Input Token → [Fixed Update Rules] → Hidden State (blurs all inputs equally)

### Selective State Space Model (Mamba)

Input Token $(x_t)$ → Projections → Dynamic Rules: $(\Delta_t, B_t, C_t)$

Update Hidden State $h_t$ selectively:

- "Is this punctuation?" → Set $\Delta_t$ small (skip / forget)
- "Is this a key entity?" → Set $\Delta_t$ large (store in memory)

## In Plain Terms

- $\Delta$ (Delta / Step Size): Acts like a gate or dwell time. A large $\Delta$ means "pay close attention to this token and write it to state memory," while a small $\Delta$ means "ignore this token, pass the previous state forward".
- $B$ and $C$: Control how much of the new input vector goes into the hidden state and how much of the hidden state is read out to produce the output.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SingleStepSelectiveSSM(nn.Module):
    """
    Conceptual implementation of Mamba's core Selective SSM mechanism 
    for 1 token step (autoregressive mode).
    """
    def __init__(self, d_model: int = 64, d_state: int = 16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # Fixed base state-transition parameters
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_model, 1)))
        
        # Projections to generate input-dependent parameters (Selective Mechanism)
        self.x_proj = nn.Linear(d_model, d_state + d_state + d_model, bias=False)
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)

        # Output projection
        self.out_proj = nn.Linear(d_model, d_model)

    def forward_step(self, x_t: torch.Tensor, h_prev: torch.Tensor):
        """
        x_t: [Batch, d_model] - Current input token vector
        h_prev: [Batch, d_model, d_state] - Hidden state from previous step
        """
        # 1. Project input token to derive B, C, and raw delta
        proj = self.x_proj(x_t) # [Batch, 2*d_state + d_model]
        B, C, dt_raw = torch.split(proj, [self.d_state, self.d_state, self.d_model], dim=-1)

        # 2. Compute dynamic input-dependent step-size Delta
        delta = F.softplus(self.dt_proj(dt_raw)) # [Batch, d_model]

        # 3. Discretize A and B based on dynamic delta
        A = -torch.exp(self.A_log) # [d_model, d_state]
        
        # A_bar = exp(delta * A) -> [Batch, d_model, d_state]
        A_bar = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0))
        
        # B_bar = delta * B -> [Batch, d_model, d_state]
        B_bar = delta.unsqueeze(-1) * B.unsqueeze(1)

        # 4. State Update: h_t = A_bar * h_prev + B_bar * x_t
        x_expanded = x_t.unsqueeze(-1) # [Batch, d_model, 1]
        h_next = A_bar * h_prev + B_bar * x_expanded # [Batch, d_model, d_state]

        # 5. Readout Output: y_t = sum(C * h_next)
        y_t = torch.sum(h_next * C.unsqueeze(1), dim=-1) # [Batch, d_model]

        return self.out_proj(y_t), h_next

if __name__ == "__main__":
    bsz, d_model, d_state = 2, 64, 16
    ssm = SingleStepSelectiveSSM(d_model=d_model, d_state=d_state)
    
    x_token = torch.randn(bsz, d_model)
    h_state = torch.zeros(bsz, d_model, d_state)

    y_out, h_next = ssm.forward_step(x_token, h_state)
    print("Output shape:", y_out.shape)   # [2, 64]
    print("Updated state shape:", h_next.shape) # [2, 64, 16]

Output shape: torch.Size([2, 64])
Updated state shape: torch.Size([2, 64, 16])
